|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Quantization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: quantize it, then find the speed you lost<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(dev)

Quantize the weights, then find out why it did not make anything faster.

Stages 18 and 18b. The first half is arithmetic and the second half is a
measurement that should annoy you.

# Exercise 1: INT8, per output channel

In [ ]:
def quantize_int8(W):
  """W (out, in) -> (int8 tensor, float scales (out,)). Symmetric,
  one scale per OUTPUT channel. Clamp the scale away from zero or an
  all-zero row gives you inf."""
  scales = 
  q = 
  return q, scales.float()

def dequantize_int8(q, scales):
  return 

W = torch.randn(512, 1024, device=dev)
q, s = quantize_int8(W)
assert q.dtype == torch.int8 and s.shape == (512,)
rel = ((dequantize_int8(q,s) - W).abs().mean()/W.abs().mean()).item()
print(f'mean relative error {rel:.4f}')
print(f'bytes: {W.numel()*4/1e6:.2f} MB fp32 -> {(q.numel()+s.numel()*4)/1e6:.2f} MB')

# Exercise 2: why per channel and not per tensor

In [ ]:
# one row with enormous weights, which is what a real model has
Wo = W.clone(); Wo[0] *= 100.0

# quantize with a SINGLE scale for the whole tensor
st = 
err_tensor = 

# and with your per-channel version
qc, sc = quantize_int8(Wo)
err_chan = 

print(f'per tensor  {err_tensor:.4f}')
print(f'per channel {err_chan:.4f}   ({err_tensor/err_chan:.0f}x better)')

# Exercise 3: now time it

Half the bytes should be most of half the time. Check.

In [ ]:
if dev == 'cuda':
  Wb = torch.randn(4096, 4096, device=dev, dtype=torch.bfloat16)
  qb, sb = quantize_int8(Wb.float())
  sbb = sb.to(torch.bfloat16)
  x = torch.randn(1, 4096, device=dev, dtype=torch.bfloat16)

  # time the plain bf16 matmul, and then the 'quantized' version that
  # rebuilds a bf16 weight before multiplying
  bf16    = 
  unfused = 

  print(f'bf16 matmul:             {bf16:7.3f} ms')
  print(f'dequantize then matmul:  {unfused:7.3f} ms  ({unfused/bf16:.1f}x SLOWER)')

### Before you open the solution

1. Exercise 3 should show the quantized version LOSING. Count the bytes
   each version moves through memory, and check your count against the
   ratio you measured.
2. The scale is one number per output channel. Write the dot product
   with the scale factored out of the sum. How many multiplies does
   applying it cost now, per output?
3. What would have to be true about the scales for that factoring to be
   impossible?